# 01 - Exploring the synthetic data and checking the patterns
**Goal:** prove that the patterns we promised in Part 1 are really in the data, and see what the cleaning step fixes.

Plain-English reminder: *EDA* (exploratory data analysis) just means 'look at the data with tables and charts before you build anything'.

In [ ]:
import sys, json, warnings; sys.path.insert(0, ".."); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.db import get_engine
from src.part2_pipeline import load_raw
from src.features import clean_metrics, compute_features
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25,
                     "figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 11})
BLUE, ORANGE, GREY = "#2f6db5", "#d9822b", "#8a8f98"
members, metrics, engagement = load_raw(get_engine())
truth = pd.read_csv("../data/synthetic_ground_truth.csv")

## 1. Size of the data
500 members x 90 days = 45,000 rows in each of the two daily tables.

In [ ]:
print(len(members), len(metrics), len(engagement)); members.head()

## 2. Pattern: weekday vs weekend
Desk workers walk more and sleep longer on weekends; field workers walk less; shift workers have no weekend sleep bump.

In [ ]:
mc, report = clean_metrics(metrics)
d = mc.merge(members[["member_id","job_type"]], on="member_id")
d["weekend"] = pd.to_datetime(d["date"]).dt.dayofweek >= 5
t = d.groupby(["job_type","weekend"])[["steps","sleep_hours"]].mean().round(1).unstack()
display(t)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for a, col, title in zip(ax, ["steps","sleep_hours"], ["Average steps","Average sleep (hours)"]):
    p = d.groupby(["job_type","weekend"])[col].mean().unstack()
    x = np.arange(len(p)); a.bar(x-.18, p[False], .36, color=BLUE, label="weekday"); a.bar(x+.18, p[True], .36, color=ORANGE, label="weekend")
    a.set_xticks(x); a.set_xticklabels(p.index); a.set_title(title); a.legend(frameon=False)
plt.tight_layout()

## 3. Pattern: shift workers sleep irregularly
The spread (standard deviation) of sleep is about twice as large for shift workers.

In [ ]:
print(d.groupby("job_type")["sleep_hours"].std().round(2))
fig, ax = plt.subplots(figsize=(7,3)); 
for j, c in [("desk", BLUE), ("shift", ORANGE)]:
    ax.hist(d.loc[d.job_type==j, "sleep_hours"].dropna(), bins=40, alpha=.6, color=c, label=j, density=True)
ax.set_title("Nightly sleep: shift workers are far more spread out"); ax.set_xlabel("hours"); ax.legend(frameon=False)

## 4. Pattern: ~15% of members gradually disengage
Average daily app opens by group (disengaging members vs the rest). The hidden answer key is only used here, to check our synthetic data.

In [ ]:
e = engagement.merge(truth[["member_id","_disengaged"]], on="member_id")
e["day"] = (pd.to_datetime(e["date"]) - pd.Timestamp("2026-07-01")).dt.days + 1
g = e.groupby(["day","_disengaged"])["app_opens"].mean().unstack().rolling(5, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(8,3.3)); ax.plot(g.index, g[False], color=GREY, label="everyone else"); ax.plot(g.index, g[True], color=ORANGE, label="disengaging members")
ax.axvline(76.5, ls="--", color="k", alpha=.4); ax.text(77, ax.get_ylim()[1]*.9, "risk label window", fontsize=8)
ax.set_title("Average app opens per day (5-day smoothing)"); ax.set_xlabel("day"); ax.legend(frameon=False)
print("disengaging members:", int(truth["_disengaged"].sum()), "of", len(truth))

## 5. Pattern: missing wearable syncs, and outliers to clean


In [ ]:
print(json.dumps(report, indent=2))
raw = metrics[["steps","sleep_hours","resting_hr","active_minutes"]]
print("\nRaw extremes (before cleaning):"); display(raw.describe().loc[["min","max"]])
print("After cleaning:"); display(mc[["steps","sleep_hours","resting_hr","active_minutes"]].describe().loc[["min","max"]])
sync = (metrics.assign(miss=metrics.wearable_synced=="no").groupby("member_id")["miss"].mean())
fig, ax = plt.subplots(figsize=(6,3)); ax.hist(sync*100, bins=25, color=BLUE); ax.set_title("Share of days without wearable sync, per member (%)")

## 6. Cleaning rules (also in the README)
1. A day with no sync stays **missing** - never replaced by 0 (a 0 would look like 'did nothing'). Averages simply skip missing days.
2. A value outside a physically possible range is **removed** (set to missing): steps 0-60,000; active minutes 0-600; sleep 2-14 h; resting HR 35-120 bpm.

Why remove instead of cap? A capped number (e.g. 150,000 steps -> 60,000) is still a made-up number.